# 01 · Data Collection — Scraping ufcstats.com

**Goal of this notebook:** build the *raw* dataset for our UFC fight-outcome model by scraping
[ufcstats.com](http://ufcstats.com), the official UFC statistics site.

We collect **four raw tables** and save them to `../data/` untouched:

| File | Grain (one row per…) | Key columns |
|---|---|---|
| `raw_events.csv` | event | `event_id, name, date, location` |
| `raw_fights.csv` | fight | `fight_id, event_id, weight_class, fighter_a_id, fighter_b_id, winner_id, method, round, time, referee` |
| `raw_fight_stats.csv` | fighter **per** fight (2 rows/fight) | knockdowns, sig. strikes (total + by target + by position), total strikes, takedowns, sub attempts, reversals, control time |
| `raw_fighters.csv` | fighter | `fighter_id, name, height, reach, stance, dob, record` |

The `record` column is the fighter's **full professional W-L-D** from the page header — display-only, never
a model feature (see Stage 3 for why). Everything else is raw fight data.

### ⚠️ The one rule for this notebook: collect raw, derive nothing

We store values **exactly as scraped**. No averages, no rolling form, no win streaks, no per-minute
rates. All of that is *feature engineering* and lives in a later notebook — and for good reason:

> The per-fight stats here (strikes landed, takedowns, control time) describe **the fight we are trying
> to predict**. Feeding them to a model as inputs is textbook **data leakage**. Later, every model
> feature must be built only from a fighter's history *strictly before* the fight date. We keep the raw
> stats now so we *can* build those point-in-time histories later — but we never feed a fight's own
> stats into its own prediction.

That is also why we scrape each fighter's **full fight history** (Stage 3): it lets us reconstruct
"what did we know about this fighter the day *before* this fight?".

## 1 · Setup

This scraper needs nothing exotic — just `requests`, `beautifulsoup4`, `lxml`, `pandas`, and `tqdm`.

**Notably, we do _not_ need Playwright / a headless browser.** An earlier plan assumed we would, because
the site shows a *"Checking your browser…"* page to plain HTTP clients. It turns out that wall is a
small JavaScript **proof-of-work** puzzle we can solve in a few lines of Python — see Section 2. That
keeps the whole pipeline lightweight (no ~150 MB Chromium download, no async browser plumbing).

Run the install cell once if these packages are missing, then restart the kernel.

In [ ]:
# Run once if needed, then restart the kernel. (Leave commented if your venv already has these.)
# %pip install requests beautifulsoup4 lxml pandas tqdm

In [ ]:
import re
import time
import random
import hashlib
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# ── Configuration ────────────────────────────────────────────────
BASE = "http://ufcstats.com"

# Where things land. Resolve relative to this notebook (notebooks/ -> ../data).
#   data/raw/        → the four raw CSV tables (the dataset we hand to preprocessing)
#   data/_html_cache → raw HTML cache so reruns are instant and polite
DATA_DIR = Path("..") / "data"
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "_html_cache"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_CSV   = RAW_DIR / "raw_events.csv"
FIGHTS_CSV   = RAW_DIR / "raw_fights.csv"
STATS_CSV    = RAW_DIR / "raw_fight_stats.csv"
FIGHTERS_CSV = RAW_DIR / "raw_fighters.csv"

# Politeness: random pause between live requests so we don't hammer the server.
MIN_DELAY, MAX_DELAY = 1.0, 2.0

# How many of the most-recent events to scrape.
#   None  → full historical scrape (~780 events; a few hours at the delay above). [default]
#   5     → quick demo that runs in minutes — set an int here for a fast smoke test.
# The scrape is resumable either way, so you can start small and re-run with None later.
MAX_EVENTS = None

## 2 · Getting past the "Checking your browser…" wall

If you fetch `http://ufcstats.com/statistics/events/completed` with plain `requests`, you get **HTTP 200**
— but the page title is literally `Loading…` and there is **zero** fight data. The real content only
appears after some JavaScript runs. That JS is a **proof-of-work (PoW) challenge**:

1. The page embeds a random `nonce` and a difficulty `target` (a number of leading hex zeros).
2. Your browser brute-forces an integer `n` such that `sha256(f"{nonce}:{n}")` starts with that many zeros.
3. It `POST`s `{nonce, n}` to `/__c`. The server checks the proof and sets a cookie (`_fmc`).
4. The page reloads — now *with* the cookie — and the real HTML is served.

This is a cost/throttling mechanism, **not** a true browser-fingerprint check, so we can reproduce all
four steps in Python. The difficulty is tiny (2 leading zeros ≈ a few hundred hashes — milliseconds),
and the `_fmc` cookie, once stored in our `requests.Session`, is reused for every subsequent page. We
solve the puzzle **once per session**, then scrape normally.

_Note: this is a polite, low-volume educational scrape of publicly available stats. We add delays and
cache pages so we never re-request what we already have._

In [ ]:
# A single Session carries cookies across every request once we've solved the PoW.
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (educational research scraper)"})


def _solve_pow(challenge_html: str) -> None:
    """Solve the SHA-256 proof-of-work and POST it so the session gets the access cookie."""
    nonce = re.search(r'nonce="([0-9a-f]+)"', challenge_html).group(1)
    zeros = int(re.search(r'target=new Array\((\d+)\+1\)', challenge_html).group(1))
    target = "0" * zeros
    n = 0
    while not hashlib.sha256(f"{nonce}:{n}".encode()).hexdigest().startswith(target):
        n += 1
    session.post(f"{BASE}/__c", data={"nonce": nonce, "n": n}, timeout=30)


def _get_html(url: str, tries: int = 4) -> str:
    """Fetch rendered HTML, solving the PoW challenge if shown and retrying transient
    network errors (connection resets, timeouts) with exponential backoff. This is what
    makes an unattended full scrape complete reliably instead of leaving random gaps."""
    solved = False
    for attempt in range(tries):
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))   # be polite
        try:
            resp = session.get(url, timeout=30)
        except requests.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 ** attempt)                        # back off, then retry
            continue
        if "Checking your browser" in resp.text and not solved:
            _solve_pow(resp.text)                           # get the access cookie…
            solved = True
            continue                                        # …and refetch, now cookied
        return resp.text
    raise RuntimeError(f"failed to fetch after {tries} tries: {url}")


def get_soup(url: str, use_cache: bool = True) -> BeautifulSoup:
    """Return parsed HTML for a URL, caching the raw HTML on disk for fast, polite reruns."""
    cache_file = CACHE_DIR / (re.sub(r"[^a-z0-9]+", "_", url.lower()).strip("_")[-120:] + ".html")
    if use_cache and cache_file.exists():
        return BeautifulSoup(cache_file.read_text(encoding="utf-8"), "lxml")
    html = _get_html(url)
    if use_cache:
        cache_file.write_text(html, encoding="utf-8")
    return BeautifulSoup(html, "lxml")

**Quick smoke test** — solve the challenge and confirm we get real data instead of the `Loading…` page.
The events list page should report hundreds of events.

In [ ]:
_test = get_soup(f"{BASE}/statistics/events/completed?page=all", use_cache=False)
_event_anchors = [a for a in _test.select("a.b-link") if "event-details" in (a.get("href") or "")]
print("Page title:", _test.title.text.strip())
print("Events found:", len(_event_anchors))
print("Newest event:", _event_anchors[0].get_text(strip=True))

## 3 · Small parsing helpers

Two utilities we reuse everywhere:

- **`url_id`** — every ufcstats URL ends in a hex id (e.g. `.../fight-details/423f32dd018ac218`). That id
  is our stable primary key for events, fights, and fighters.
- **`split_landed_attempted`** — strike/takedown cells read like `"15 of 25"`. We split them into two
  numeric columns (`landed`, `attempted`). This is *structural* parsing, not feature engineering — we're
  just unpacking one cell into the two raw numbers it already contains.

In [ ]:
def url_id(url: str) -> str:
    """Last path segment of a ufcstats URL — the stable hex id we use as a primary key."""
    return url.rstrip("/").split("/")[-1]


def split_landed_attempted(text: str):
    """'15 of 25' -> (15, 25). Returns (None, None) for '---' or anything unparseable."""
    m = re.match(r"(\d+)\s+of\s+(\d+)", text.strip())
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

## 4 · Stage 1 — Events

Two functions:

- **`get_event_links()`** — the completed-events index (`?page=all` returns them all on one page).
  We keep every `event-details` link, newest first.
- **`parse_event(url)`** — open one event page and read its name (`h2.b-content__title`), `Date:` and
  `Location:` list items, and the **fight links**. Each fight row on an event page carries the fight URL
  in a `data-link` attribute (`tr.b-fight-details__table-row[data-link]`).

In [ ]:
def get_event_links() -> list[str]:
    """All completed-event URLs, newest first."""
    soup = get_soup(f"{BASE}/statistics/events/completed?page=all", use_cache=False)
    seen, links = set(), []
    for a in soup.select("a.b-link"):
        href = a.get("href") or ""
        if "event-details" in href and href not in seen:
            seen.add(href)
            links.append(href)
    return links


def parse_event(url: str):
    """Return (event_dict, [fight_urls]) for one event page."""
    soup = get_soup(url)
    name = soup.select_one("h2.b-content__title").get_text(strip=True)
    date = location = None
    for li in soup.select("li.b-list__box-list-item"):
        t = li.get_text(" ", strip=True)
        if t.startswith("Date:"):
            date = t.split("Date:", 1)[1].strip()
        elif t.startswith("Location:"):
            location = t.split("Location:", 1)[1].strip()
    fight_urls = [r.get("data-link")
                  for r in soup.select("tr.b-fight-details__table-row[data-link]")]
    event = {"event_id": url_id(url), "name": name, "date": date, "location": location}
    return event, fight_urls

## 5 · Stage 2 — Fights & per-fighter stats (the core)

A fight page has three parts we care about:

1. **The two fighters** — each `div.b-fight-details__person` holds a result flag
   (`i.b-fight-details__person-status` = `W`/`L`/`D`/`NC`) and a link to the fighter (whose id we keep).
   By convention person 0 is the red corner (`fighter_a`), person 1 the blue corner (`fighter_b`).
   ⚠️ *Remember the corner-leakage rule later: red wins ~57% purely by promoter convention, so we'll build
   symmetric features rather than letting the model memorise "red wins".*
2. **The outcome line** — `Method: … Round: … Time: … Referee: …` (one `p.b-fight-details__text`).
3. **Two stat tables** (`table.b-fight-details__table`):
   - Table 0 = **Totals**: KD, Sig. str. (`X of Y`), Total str., Td, Sub. att, Rev., Ctrl.
   - Table 1 = **Significant strikes** broken out by target (Head/Body/Leg) and position (Distance/Clinch/Ground).

Each table also has a *per-round* row beneath the totals row — we take **row 0 (the full-fight totals)**
only. Within a cell, the two fighters' values are two separate `<p>` tags, so `<p>[i]` gives fighter *i*'s value.

In [ ]:
def _cell(cells, col, fighter_idx):
    """Text of fighter `fighter_idx`'s value in column `col` (each cell has one <p> per fighter)."""
    ps = cells[col].select("p")
    return ps[fighter_idx].get_text(strip=True) if len(ps) > fighter_idx else ""


def parse_fight(url: str, event_id: str):
    """Return (fight_dict, [stat_rows]) for one fight. stat_rows has one entry per fighter."""
    soup = get_soup(url)

    # --- fighters & result -------------------------------------------------
    fighters = []
    for person in soup.select("div.b-fight-details__person"):
        a = person.select_one("a.b-link")
        status = person.select_one("i.b-fight-details__person-status").get_text(strip=True)
        fighters.append({"id": url_id(a.get("href")), "status": status})
    winner_id = next((f["id"] for f in fighters if f["status"] == "W"), None)  # None = draw/NC

    # --- outcome line ------------------------------------------------------
    # Upcoming/unfought bouts have no result line (just a 'tale of the tape').
    # Signal 'skip cleanly' with (None, []) so the caller can tell them apart
    # from genuine parse errors.
    info_el = soup.select_one("p.b-fight-details__text")
    if info_el is None:
        return None, []
    info = info_el.get_text(" ", strip=True)
    def grab(pattern):
        m = re.search(pattern, info)
        return m.group(1).strip() if m else None

    fight = {
        "fight_id": url_id(url),
        "event_id": event_id,
        "weight_class": soup.select_one("i.b-fight-details__fight-title").get_text(" ", strip=True),
        "fighter_a_id": fighters[0]["id"],
        "fighter_b_id": fighters[1]["id"],
        "winner_id": winner_id,
        "method": grab(r"Method:\s*(.+?)\s+Round:"),
        "round": grab(r"Round:\s*(\d+)"),
        "time": grab(r"Time:\s*(\d+:\d+)"),
        "referee": grab(r"Referee:\s*(.+)$"),
    }

    # --- per-fighter stat tables (full-fight totals = row 0) ---------------
    stat_rows = []
    tables = soup.select("table.b-fight-details__table")
    if len(tables) >= 2:
        tot = tables[0].select("tbody tr")[0].select("td")   # KD, Sig, Sig%, Tot, Td, Td%, Sub, Rev, Ctrl
        sig = tables[1].select("tbody tr")[0].select("td")   # Sig, Sig%, Head, Body, Leg, Dist, Clinch, Ground
        for i, f in enumerate(fighters):
            sig_l, sig_a       = split_landed_attempted(_cell(tot, 2, i))
            tot_l, tot_a       = split_landed_attempted(_cell(tot, 4, i))
            td_l, td_a         = split_landed_attempted(_cell(tot, 5, i))
            head_l, head_a     = split_landed_attempted(_cell(sig, 3, i))
            body_l, body_a     = split_landed_attempted(_cell(sig, 4, i))
            leg_l, leg_a       = split_landed_attempted(_cell(sig, 5, i))
            dist_l, dist_a     = split_landed_attempted(_cell(sig, 6, i))
            clinch_l, clinch_a = split_landed_attempted(_cell(sig, 7, i))
            grnd_l, grnd_a     = split_landed_attempted(_cell(sig, 8, i))
            stat_rows.append({
                "fight_id": fight["fight_id"], "fighter_id": f["id"],
                "knockdowns": _cell(tot, 1, i),
                "sig_str_landed": sig_l, "sig_str_att": sig_a,
                "total_str_landed": tot_l, "total_str_att": tot_a,
                "takedowns_landed": td_l, "takedowns_att": td_a,
                "sub_att": _cell(tot, 7, i), "reversals": _cell(tot, 8, i),
                "control_time": _cell(tot, 9, i),
                "sig_head_landed": head_l, "sig_head_att": head_a,
                "sig_body_landed": body_l, "sig_body_att": body_a,
                "sig_leg_landed": leg_l, "sig_leg_att": leg_a,
                "sig_distance_landed": dist_l, "sig_distance_att": dist_a,
                "sig_clinch_landed": clinch_l, "sig_clinch_att": clinch_a,
                "sig_ground_landed": grnd_l, "sig_ground_att": grnd_a,
            })
    return fight, stat_rows

## 6 · Stage 3 — Fighter bios

Finally, one row per fighter: name, height, reach, stance, date of birth, and the page-header
**`record`**. A fighter's page URL is simply `BASE/fighter-details/{fighter_id}`, so we can rebuild it
from any id we collected in Stage 2 — no need to store URLs anywhere.

The header `record` is the fighter's **full professional W-L-D** — including pre-UFC bouts (WEC, Bellator,
regional shows) that never appear as fight rows on ufcstats. We keep it purely for **display** (the app
quotes a fighter's real record, e.g. `27-1`, not just their `17-1` UFC-only tally). It is **not** a model
feature: it's a present-day aggregate that would leak future results into past fights, so every model
input is still built from the strictly-prior UFC fight history.

(The page also lists career averages like SLpM and a full fight history; we deliberately **skip the
career averages** — those are pre-computed aggregates and using them as features would risk leakage. The
raw per-fight rows we already saved let us compute point-in-time history ourselves later.)

In [ ]:
def parse_fighter(fighter_id: str):
    """Return a bio dict for one fighter id.

    `record` is the page-header 'W-L-D' — the fighter's FULL professional record,
    including pre-UFC bouts that never appear as fight rows on ufcstats (e.g. a
    fighter listed 27-1 here may have only ~17 UFC fights in raw_fights.csv). It's
    stored for DISPLAY only; every model feature is still built from the UFC fight
    history, so this is not a leakage source.
    """
    soup = get_soup(f"{BASE}/fighter-details/{fighter_id}")
    rec_el = soup.select_one("span.b-content__title-record")
    record = rec_el.get_text(strip=True).split("Record:", 1)[-1].strip() if rec_el else None
    bio = {
        "fighter_id": fighter_id,
        "name": soup.select_one("span.b-content__title-highlight").get_text(strip=True),
        "height": None, "reach": None, "stance": None, "dob": None,
        "record": record,
    }
    labels = {"Height:": "height", "Reach:": "reach", "STANCE:": "stance", "DOB:": "dob"}
    for li in soup.select("li.b-list__box-list-item"):
        t = li.get_text(" ", strip=True)
        for lbl, key in labels.items():
            if t.startswith(lbl):
                bio[key] = t.split(lbl, 1)[1].strip()
    return bio

## 7 · Resumability & incremental saving

Scraping hundreds of events takes a while and *will* get interrupted. Two cheap safeguards make reruns
safe and fast:

- **HTML cache** (Section 2) — any page we've fetched is read from disk, so a rerun never re-hits the network for it.
- **Append + skip** — we write each table with `mode="a"`, and on startup read back the ids already saved.
  Events whose `event_id` is already in `raw_fights.csv` are skipped; fighters already in `raw_fighters.csv`
  are skipped. We commit a whole event's rows in one shot **after** it parses cleanly, so a crash mid-event
  never leaves half a card behind to be duplicated on rerun.

In [ ]:
def append_rows(path: Path, rows: list[dict]) -> None:
    """Append dict rows to a CSV, writing the header only when the file is new."""
    if not rows:
        return
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)


def done_ids(path: Path, column: str) -> set[str]:
    """Set of already-saved ids in `column`, or empty set if the file doesn't exist yet."""
    if path.exists():
        return set(pd.read_csv(path, usecols=[column])[column].astype(str))
    return set()

## 8 · Run the scrape — events → fights → stats

Now the orchestration. For each event (newest first, capped at `MAX_EVENTS` for the demo) we parse the
card, then every fight on it, then commit the event + its fights + its stats together. Every page is
wrapped in `try/except` so a single bad page is **logged and skipped** rather than killing the whole run.

Note the newest "event" is often the *next, unfought* card. Those bouts have no result yet, so
`parse_fight` returns `None` and we skip them cleanly (counted separately from genuine parse failures) —
we never want unfought bouts in a training set anyway.

Set `MAX_EVENTS = None` in Section 1 and re-run for the full history.

In [ ]:
event_links = get_event_links()
print(f"{len(event_links)} completed events available.")

todo = event_links if MAX_EVENTS is None else event_links[:MAX_EVENTS]
done_events = done_ids(FIGHTS_CSV, "event_id")
skipped = []
upcoming = 0   # bouts with no result yet (e.g. the next, unfought event)

for ev_url in tqdm(todo, desc="events"):
    eid = url_id(ev_url)
    if eid in done_events:
        continue
    try:
        event, fight_urls = parse_event(ev_url)
    except Exception as e:
        skipped.append((ev_url, f"event parse: {e}"))
        continue

    fights, stats = [], []
    for f_url in fight_urls:
        try:
            fight, stat_rows = parse_fight(f_url, eid)
            if fight is None:          # upcoming/unfought bout — nothing to record
                upcoming += 1
                continue
            fights.append(fight)
            stats.extend(stat_rows)
        except Exception as e:
            skipped.append((f_url, f"fight parse: {e}"))

    # Commit the event atomically, but only once it has >=1 completed fight.
    # (An all-upcoming card writes nothing and is retried on a later run.)
    if fights:
        append_rows(FIGHTS_CSV, fights)
        append_rows(STATS_CSV, stats)
        append_rows(EVENTS_CSV, [event])
        done_events.add(eid)

print(f"Done. {upcoming} unfought bouts skipped (no result yet), "
      f"{len(skipped)} pages failed to parse.")
for url, why in skipped[:10]:
    print("  FAILED", why, "->", url)

## 9 · Run the scrape — fighter bios

Every fighter we need is referenced by `fighter_a_id` / `fighter_b_id` in `raw_fights.csv`. We take that
set, drop anyone already saved, and scrape the rest. Because we derive the list from the saved fights
file (not from in-memory state), this stage resumes correctly even on a fresh kernel.

In [ ]:
fights_df = pd.read_csv(FIGHTS_CSV)
needed = set(fights_df["fighter_a_id"].astype(str)) | set(fights_df["fighter_b_id"].astype(str))
needed -= done_ids(FIGHTERS_CSV, "fighter_id")
print(f"{len(needed)} fighters to scrape.")

bios, skipped_f = [], []
for fid in tqdm(sorted(needed), desc="fighters"):
    try:
        bios.append(parse_fighter(fid))
    except Exception as e:
        skipped_f.append((fid, str(e)))
    if len(bios) >= 25:                 # flush periodically so progress survives a crash
        append_rows(FIGHTERS_CSV, bios)
        bios = []
append_rows(FIGHTERS_CSV, bios)
print(f"Done. {len(skipped_f)} fighters skipped.")

## 10 · Sanity checks

A quick look at shapes, a sample row from each table, and a couple of integrity checks before we hand off
to preprocessing.

In [ ]:
events    = pd.read_csv(EVENTS_CSV)
fights    = pd.read_csv(FIGHTS_CSV)
stats     = pd.read_csv(STATS_CSV)
fighters  = pd.read_csv(FIGHTERS_CSV)

for name, df in [("events", events), ("fights", fights),
                 ("fight_stats", stats), ("fighters", fighters)]:
    print(f"{name:12s} {df.shape}")

# Integrity: no duplicate keys, two stat rows per fight, every fighter has a bio.
dupes = (int(events.event_id.duplicated().sum())
         + int(fights.fight_id.duplicated().sum())
         + int(fighters.fighter_id.duplicated().sum()))
per_fight = stats.groupby("fight_id").size()
missing_bio = (set(fights["fighter_a_id"]) | set(fights["fighter_b_id"])) - set(fighters["fighter_id"])

print("\nduplicate primary keys           :", dupes)
print("fights with != 2 stat rows       :", int((per_fight != 2).sum()))
print("fighters referenced w/o a bio    :", len(missing_bio))

assert dupes == 0, "duplicate rows present — re-run the finalize cell"
assert len(missing_bio) == 0, "some fighters lack a bio — re-run the fighter-scrape cell"
print("\nAll integrity checks passed ✔")

In [ ]:
stats.head(4)

## 11 · Recap & next steps

We now have four raw tables in `../data/`:

- `raw_events.csv`, `raw_fights.csv`, `raw_fight_stats.csv`, `raw_fighters.csv`

**What we did NOT do (on purpose):** no averages, no rolling form, no derived rates, no corner
adjustments. Every value is as it appeared on the page.

**Next notebook — feature engineering — must respect two rules:**

1. **Point-in-time only.** Build each fighter's features from fights *strictly before* the target fight's
   date (career & recent aggregates as-of that date). Never use a fight's own stats to predict it.
2. **No corner leakage.** Red wins ~57% by promoter convention. Use symmetric fighter-vs-fighter
   *difference* features and/or randomise corner assignment so the model learns skill, not seating.

Then: a **temporal** train/test split (train on older fights, test on the most recent ~1–2 years),
evaluated on log-loss + calibration, and benchmarked against the "always pick the favourite" baseline —
on the way to the Logistic Regression vs XGBoost comparison that is this project's centrepiece.